# Human-in-the-Loop：行動前閘道、風險分層與稽核日誌

本課程的 README 以一段簡短的程式碼片段介紹了 Human-in-the-Loop，該片段會在代理已產生回應後，要求使用者進行 `APPROVE` 或 `REJECT`。這種模式是個不錯的起點，但實務上的 HITL 實作通常還需要三個額外的部分：

1. 一個在代理執行具風險步驟<strong>之前</strong>運作的<strong>行動前閘道</strong>，以保持成本、不可逆性和延遲在可控範圍內。
2. <strong>風險分層</strong>，使低風險行動自動執行、中風險行動批次批准，且只有高風險行動才需要等待人類確認。
3. 一個<strong>稽核日誌加修正迴圈</strong>，以 JSONL 格式記錄每個閘道決策，且拒絕時用結構化原因重新提示代理，而非僅印出 `Revising...`。

本筆記本基於與 `06-system-message-framework.ipynb` 相同的原始元件構建上述每個功能。它可於 `DEMO_MODE = True` 下全程執行（無需互動輸入），或於 `DEMO_MODE = False` 時使用真實的 `input()` 提示。注意：在 DEMO_MODE 中，第三目標的重試是腳本化的，因此迴圈機制可端對端觀察。真正的基於修正的重新分類則需 `DEMO_MODE = False` 且有人操作。

**非本課程範圍（其他課程處理）：** 認證及存取控制（課程 06 README 中的威脅 2）、工具呼叫中介軟體（課程 14 MAF 深入探討）、多代理辯論模式。

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: 預先動作閘門

README 的 HITL 範例先呼叫代理人，然後再請使用者批准輸出。那是一個<strong>動作後</strong>流程。代理人已經執行，所以 LLM 呼叫成本已經付出，且任何副作用（寄送電子郵件、寫入資料庫列、發布評論）也已經發生。

一個<strong>動作前</strong>流程會在代理人執行風險步驟之前插入閘門。代理人提出動作，閘門決定是否執行，只有在批准時才會發生副作用。

| 方面 | 動作後批准（README 範例） | 動作前閘門（本筆記） |
|---|---|---|
| 何時執行批准？ | 代理人產生輸出後 | 任何副作用執行前 |
| 拒絕時的 LLM 成本 | 已付出 | 只為提案付費，不為動作付費 |
| 不可逆的副作用 | 可能（動作已經發生） | 避免發生 |
| 審核清晰度 | 批准為列印語句 | 批准為帶有時間戳、動作、原因的 JSON 紀錄 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: 風險分級

並非每個操作都需要人工批准。對公共 API 進行唯讀查詢與發送客戶電子郵件的重要性不同。將兩者視為相同會浪費操作員的注意力並使代理變慢。

一個簡單的 3 級模型：

| Tier | 範例 | 批准流程 |
|---|---|---|
| `low` (唯讀) | 搜尋知識庫、查詢航班選項、擷取公共網頁 | 自動執行，紀錄以供稽核 |
| `medium` (輕量變更) | 快取結果、計數器加一、排程提醒 | 自動執行，但每日批次審查 |
| `high` (面向外部或不可逆) | 發送電子郵件、扣款、發佈至公共頻道 | 需人工批准方可執行 |

這是一種分級法。生產系統經常使用更細緻的分級（例如 AWS IAM 權限級別、基於角色的存取分層）。以下的 3 級版本是同時混合唯讀與有副作用操作的代理中最簡潔實用的版本。

下面的分類器使用關鍵字啟發式，讓示範保持確定性與低成本。在生產系統中，你會用學習式分類器或策略引擎取代它。

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: 稽核日誌與修訂迴圈

`print("Response approved.")` 並不是稽核日誌。為了建立信任，每個閘道決策都應記錄為結構化事件，日後你可以查詢、重播或附加至事件檢討。

兩個部分：

1. **只能新增的 JSONL。** 每個決策一行，包含時間戳、動作、階層、決策、原因。容易使用 grep 搜尋，且日後方便傳送至真正的日誌儲存系統。
2. **拒絕時的修訂迴圈。** 當閘道回傳 `deny` 時，代理會帶著拒絕原因自我重新提問，讓下次的提案能避開問題。

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 額外資源

其他幾個公開專案實作了這些 HITL 模式的變體。比較不同方法，找出適合您技術棧的方案：

- **LangChain** 人在迴圈中的工具包裝器（[文件](https://python.langchain.com/docs/integrations/tools/human_tools)）：即插即用的工具包裝器，可暫停執行等待人工輸入。
- **AutoGen** `UserProxyAgent`（[v0.2 文件](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop)；AutoGen v0.4 版本起已重構）：使用專門代表人在多代理對話中的代理角色。
- **Semantic Kernel** 函數過濾器（[文件](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)）：環繞每個函數呼叫執行的中介軟體，適合用來做閘道邏輯。

每個專案對這三個子模式的處理方式不同：LangChain 將其包裝成工具，AutoGen 使用代理角色，Semantic Kernel 則是使用中介軟體過濾器。在選擇自己的代理設計之前，建議完整閱讀一到兩個實作範例。

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
